# Episode 3: Matrices as Machines
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2003%20-%20Matrices.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 3, one step per section. It uses only `numpy` (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: a matrix as a rule that moves vectors, the shape rule, the linear layer h = Wu + β, stacked inputs U, and the collapse of two linear layers into one. All weights are hand-chosen and illustrative, not learned.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

## Setup
W is 2×3, β has 2 components, and U stacks the three toy words as rows (one word per row, shape n × d).

In [1]:
import numpy as np

W    = np.array([[1, 0, 0],
                 [0, 2, 1]], dtype=float)   # d_out × d = 2 × 3
beta = np.array([0, 1], dtype=float)

cat   = np.array([1, 1, 0], dtype=float)
mat   = np.array([1, 0, 0], dtype=float)
tired = np.array([0, 0, 1], dtype=float)
U = np.stack([cat, mat, tired])             # n × d = 3 × 3

print("W shape:", W.shape, "  U shape:", U.shape)

W shape: (2, 3)   U shape: (3, 3)


## The column picture
Wu is a weighted sum of the columns of W, with the entries of u as the weights.

In [2]:
u = cat
by_columns = u[0] * W[:, 0] + u[1] * W[:, 1] + u[2] * W[:, 2]
print("W @ cat           =", W @ u)
print("sum of columns    =", by_columns)
assert np.allclose(W @ u, by_columns)

W @ cat           = [1. 2.]
sum of columns    = [1. 2.]


## Step 2: apply W to each row, one word at a time
Each output component is a row of W dotted with the input.

In [3]:
for name, vec in [("cat", cat), ("mat", mat), ("tired", tired)]:
    print(name, "->", W @ vec)

assert np.allclose(W @ cat,   [1, 2])
assert np.allclose(W @ mat,   [1, 0])
assert np.allclose(W @ tired, [0, 1])

cat -> [1. 2.]
mat -> [1. 0.]
tired -> [0. 1.]


## Step 3: add the bias β = (0, 1)

In [4]:
h_cat, h_mat, h_tired = W @ cat + beta, W @ mat + beta, W @ tired + beta
print("cat   ->", h_cat)
print("mat   ->", h_mat)
print("tired ->", h_tired)
assert np.allclose(h_cat,   [1, 3])
assert np.allclose(h_mat,   [1, 1])
assert np.allclose(h_tired, [0, 2])

cat   -> [1. 3.]
mat   -> [1. 1.]
tired -> [0. 2.]


## Step 4: the batched form
Wᵀ is 3×2, so U Wᵀ is (3×3)(3×2) = 3×2. Adding β to every row reproduces Step 3 in one product.

In [5]:
batched = U @ W.T                  # (n × d)(d × d_out) = n × d_out
print("U Wᵀ shape:", batched.shape)
print(batched)
assert batched.shape == (3, 2)
assert np.allclose(batched, [[1, 2], [1, 0], [0, 1]])

H = batched + beta                 # β is added to every row (broadcast)
print("With β added:")
print(H)
assert np.allclose(H, [[1, 3], [1, 1], [0, 2]])

U Wᵀ shape: (3, 2)
[[1. 2.]
 [1. 0.]
 [0. 1.]]
With β added:
[[1. 3.]
 [1. 1.]
 [0. 2.]]


## Step 5: the shape rule, and one entry by hand
(m×k)(k×r) gives m×r. The product is undefined when the inner sizes differ.

In [6]:
A = np.array([[1, 0, 1],
              [0, 2, 0]], dtype=float)   # 2 × 3
B = np.array([[1, 0],
              [0, 1],
              [1, 1]], dtype=float)      # 3 × 2
AB = A @ B
print("AB =\n", AB, "\nshape:", AB.shape)
assert AB.shape == (2, 2)

# Entry (1, 2) in the episode's 1-based counting is AB[0, 1] here (Python counts from 0).
by_hand = A[0, 0] * B[0, 1] + A[0, 1] * B[1, 1] + A[0, 2] * B[2, 1]
print("Entry (1, 2) by hand: 1·0 + 0·1 + 1·1 =", by_hand)
assert by_hand == 1 and AB[0, 1] == 1

# The three shape checks from the episode
print((np.ones((2, 3)) @ np.ones((3, 2))).shape)   # (2×3)(3×2) -> 2×2
print((np.ones((3, 2)) @ np.ones((2, 3))).shape)   # (3×2)(2×3) -> 3×3
try:
    np.ones((2, 3)) @ np.ones((2, 3))               # (2×3)(2×3) -> undefined
except ValueError as e:
    print("(2×3)(2×3) is undefined:", "ValueError")

# Order matters: AB and BA can differ (here they do not even have the same shape)
print("AB shape", (A @ B).shape, " BA shape", (B @ A).shape)

AB =
 [[2. 1.]
 [0. 2.]] 
shape: (2, 2)
Entry (1, 2) by hand: 1·0 + 0·1 + 1·1 = 1.0
(2, 2)
(3, 3)
(2×3)(2×3) is undefined: ValueError
AB shape (2, 2)  BA shape (3, 3)


## Step 6 (preview only): two linear layers collapse into one
A second layer W′ = [1, 1] (1×2), no bias. Episode 11 develops this.

In [7]:
W2 = np.array([[1, 1]], dtype=float)       # 1 × 2

two_steps = W2 @ (W @ cat)
one_matrix = (W2 @ W)
print("W′(W cat)       =", two_steps)
print("W′W (one matrix) =", one_matrix)
print("(W′W) cat        =", one_matrix @ cat)
assert np.allclose(two_steps, [3])
assert np.allclose(one_matrix, [[1, 2, 1]])
assert np.allclose(one_matrix @ cat, [3])

# General identity with a bias: W′(Wu + β) = (W′W)u + W′β   (distributivity and associativity)
rng = np.random.default_rng(0)
for _ in range(5):
    u = rng.normal(size=3)
    lhs = W2 @ (W @ u + beta)
    rhs = (W2 @ W) @ u + W2 @ beta
    assert np.allclose(lhs, rhs)
print("W′(Wu + β) = (W′W)u + W′β holds for 5 random u.")

W′(W cat)       = [3.]
W′W (one matrix) = [[1. 2. 1.]]
(W′W) cat        = [3.]
W′(Wu + β) = (W′W)u + W′β holds for 5 random u.


## Check question
A layer takes 3-number inputs and returns 2-number outputs. (a) What shape is W? (b) If 10 token vectors are stacked as rows, what shape is the batched output?

In [8]:
print("(a) W shape:", W.shape, " which is d_out × d")
U10 = np.ones((10, 3))
print("(b) batched output shape:", (U10 @ W.T + beta).shape)
assert W.shape == (2, 3)
assert (U10 @ W.T + beta).shape == (10, 2)
print("(c) Two linear layers with no nonlinearity multiply to one matrix, so one layer does the same job.")

(a) W shape: (2, 3)  which is d_out × d
(b) batched output shape: (10, 2)
(c) Two linear layers with no nonlinearity multiply to one matrix, so one layer does the same job.


In [9]:
print("All Episode 3 checks passed.")

All Episode 3 checks passed.
